# Healthcare SQL Analysis

This notebook demonstrates SQL-based healthcare analytics using the cleaned healthcare dataset.

The analysis focuses on patient demographics, healthcare utilization, medical conditions, healthcare spending, healthcare organizations, and provider characteristics.

## Database Connection

The cleaned healthcare datasets are loaded into a SQLite database to enable SQL-based analysis.

In [10]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("healthcare.db")

In [12]:
patients = pd.read_csv("patients_clean.csv")
encounters = pd.read_csv("encounters_clean.csv")
conditions = pd.read_csv("conditions_clean.csv")
providers = pd.read_csv("providers_clean.csv")
organizations = pd.read_csv("organizations_clean.csv")

In [13]:
patients.to_sql(
    "patients",
    conn,
    if_exists="replace",
    index=False
)

encounters.to_sql(
    "encounters",
    conn,
    if_exists="replace",
    index=False
)

conditions.to_sql(
    "conditions",
    conn,
    if_exists="replace",
    index=False
)

providers.to_sql(
    "providers",
    conn,
    if_exists="replace",
    index=False
)

organizations.to_sql(
    "organizations",
    conn,
    if_exists="replace",
    index=False
)

1127

In [14]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table'
""", conn)

,name
0,patients
1,encounters
2,conditions
3,providers
4,organizations


## Patients by Age Group

In [15]:
query = """
SELECT
    age_group,
    COUNT(*) AS patient_count
FROM patients
GROUP BY age_group
ORDER BY patient_count DESC;
"""

pd.read_sql(query, conn)

,age_group,patient_count
0,65+,282
1,18-34,256
2,50-64,249
3,35-49,191
4,0-17,185


## Patients by Gender

In [16]:
query = """
SELECT
    gender,
    COUNT(*) AS patient_count
FROM patients
GROUP BY gender;
"""

pd.read_sql(query, conn)

,gender,patient_count
0,F,616
1,M,547


## Encounter Volume by Type

In [17]:
query = """
SELECT
    encounter_type,
    COUNT(*) AS total_encounters
FROM encounters
GROUP BY encounter_type
ORDER BY total_encounters DESC;
"""

pd.read_sql(query, conn)

,encounter_type,total_encounters
0,wellness,24038
1,ambulatory,20124
2,outpatient,10837
3,urgentcare,2564
4,emergency,2168
5,inpatient,1728


## Top 10 Patients by Encounter Count

In [18]:
query = """
SELECT
    patient_id,
    COUNT(*) AS encounter_count
FROM encounters
GROUP BY patient_id
ORDER BY encounter_count DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,patient_id,encounter_count
0,ef167059-cef0-12c4-49db-993ca3a20c01,1563
1,a1c127d9-d31e-36b1-2a3e-3014be9bfdf0,838
2,e23c7a6e-6ef2-3932-1361-b73b4c8a3961,705
3,8c6ae452-5f8c-9ff6-006d-c6c860acf5cd,638
4,e6178711-3936-e332-814d-19d8851e314d,497
5,139f1206-227d-4cf1-2bea-7d3154516bdf,447
6,55c5b8d3-99d0-58ad-8444-e42bb81bd5c7,412
7,562e7d26-9dfb-95f2-0815-3c20129955b8,401
8,e0b4bd84-4df2-d30c-cd67-a9daee769602,390
9,864b2fa0-78e3-88c6-2ce2-ba3aea72236c,383


## Most Common Conditions

In [19]:
query = """
SELECT
    condition_name,
    COUNT(*) AS total_cases
FROM conditions
GROUP BY condition_name
ORDER BY total_cases DESC
LIMIT 10;
"""

pd.read_sql(query, conn)


,condition_name,total_cases
0,Full-time employment (finding),13805
1,Stress (finding),5137
2,Part-time employment (finding),2426
3,Social isolation (finding),1243
4,Viral sinusitis (disorder),1233
5,Limited social contact (finding),1200
6,Not in labor force (finding),1077
7,Victim of intimate partner abuse (finding),819
8,Acute viral pharyngitis (disorder),678
9,Acute bronchitis (disorder),571


## Active vs Resolved Conditions

In [20]:
query = """
SELECT
    condition_status,
    COUNT(*) AS total_conditions
FROM conditions
GROUP BY condition_status;
"""

pd.read_sql(query, conn)

,condition_status,total_conditions
0,Active,8169
1,Resolved,29925


## Total Healthcare Spending

In [21]:
query = """
SELECT
    ROUND(SUM(total_claim_cost),2) AS total_spending
FROM encounters;
"""

pd.read_sql(query, conn)

,total_spending
0,2.550338e+08


## Cost by Encounter Type

In [22]:
query = """
SELECT
    encounter_type,
    ROUND(SUM(total_claim_cost),2) AS total_cost
FROM encounters
GROUP BY encounter_type
ORDER BY total_cost DESC;
"""

pd.read_sql(query, conn)

,encounter_type,total_cost
0,ambulatory,1.312929e+08
1,wellness,4.590036e+07
2,outpatient,3.064173e+07
3,emergency,1.718445e+07
4,inpatient,1.514765e+07
5,urgentcare,1.486678e+07


## Average Cost by Encounter Type

In [23]:
query = """
SELECT
    encounter_type,
    ROUND(AVG(total_claim_cost),2) AS avg_cost
FROM encounters
GROUP BY encounter_type
ORDER BY avg_cost DESC;
"""

pd.read_sql(query, conn)

,encounter_type,avg_cost
0,inpatient,8766.00
1,emergency,7926.41
2,ambulatory,6524.19
3,urgentcare,5798.27
4,outpatient,2827.51
5,wellness,1909.49


## Cost by Age Group

In [24]:
query = """
SELECT
    p.age_group,
    ROUND(SUM(e.total_claim_cost),2) AS total_cost
FROM encounters e
JOIN patients p
    ON e.patient_id = p.patient_id
GROUP BY p.age_group
ORDER BY total_cost DESC;
"""

pd.read_sql(query, conn)

,age_group,total_cost
0,65+,96990524.94
1,18-34,55206229.47
2,50-64,50948573.17
3,35-49,48071165.61
4,0-17,3817334.89


## Top Organizations by Utilization

In [25]:
query = """
SELECT
    organization_name,
    organization_utilization
FROM organizations
ORDER BY organization_utilization DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,organization_name,organization_utilization
0,NORTH SHORE MEDICAL CENTER -,3954
1,LOWELL GENERAL HOSPITAL,3852
2,LAHEY HOSPITAL & MEDICAL CENTER BURLINGTON,3409
3,FALMOUTH HOSPITAL,3111
4,MOUNT AUBURN HOSPITAL,2877
5,CAMBRIDGE HEALTH ALLIANCE,2706
6,MORTON HOSPITAL,2673
7,BETH ISRAEL DEACONESS HOSPITAL - PLYMOUTH,2595
8,WINCHESTER HOSPITAL,2555
9,NEWTON-WELLESLEY HOSPITAL,2380


## Provider Specialty Distribution

In [26]:
query = """
SELECT
    specialty,
    COUNT(*) AS provider_count
FROM providers
GROUP BY specialty
ORDER BY provider_count DESC
LIMIT 15;
"""

pd.read_sql(query, conn)

,specialty,provider_count
0,GENERAL PRACTICE,1128
1,PHYSICIAN ASSISTANT,542
2,NURSE PRACTITIONER,387
3,CLINICAL SOCIAL WORKER,376
4,PHYSICAL THERAPY,344
5,INTERNAL MEDICINE,312
6,FAMILY PRACTICE,259
7,CHIROPRACTIC,153
8,OPTOMETRY,151
9,CLINICAL PSYCHOLOGIST,139


# SQL Analysis Summary

The SQL analysis validated key findings from the exploratory data analysis and demonstrated the use of:

- SELECT
- GROUP BY
- ORDER BY
- LIMIT
- Aggregate Functions
- INNER JOIN

The analysis provided additional evidence of healthcare utilization patterns, patient demographics, healthcare spending trends, and provider distribution.